In [21]:
import os
import cv2
import pyewts
import fastwer
import numpy as np
from glob import glob
from tqdm import tqdm
from natsort import natsorted
from evaluate import load
from Utils import read_label, get_file_name, pad_image2, binarize_line, process_wylie_label
from Modules import OnnxInference

In [3]:
def prepare_ocr_image2(
    line_image: np.array, target_width: int = 2000, target_height: int = 80
    ):
        if len(line_image.shape) > 2:
            line_image = cv2.cvtColor(line_image, cv2.COLOR_BGR2GRAY)

        image = pad_image2(
            line_image,
            target_width=target_width,
            target_height=target_height,
            padding="black",
        )
        
        image = image.reshape((1, target_height, target_width))

        #image = (image / 127.5) - 1.0
        image = image / 255.0
        image = image.astype(np.float32)
        return image

In [4]:
converter = pyewts.pyewts()
cer_scorer = load("cer")

In [16]:
#dataset_root = "Models/OCRModels"
#dataset = "DergeTenjur"
#chckpt = "2024_1_25_6_48"


"""dataset_root = "Models/OCRModels"
dataset = "LhasaKanjur"
chckpt = "Prodigy_2023_9_13_7_59"

model_config = f"{dataset_root}/{dataset}/{chckpt}/model_config.json"
ocr_inference = OnnxInference(config_file=model_config)
"""

"""
dataset_root = "Models/OCRModels"
dataset = "LhasaKanjur"
chckpt = "Prodigy_2023_9_13_7_59/EASTER2"
"""

dataset_root = "Models/OCRModels"
dataset = "LithangKanjur"
chckpt = "2023_12_11_19_46"

model_config = f"{dataset_root}/{dataset}/{chckpt}/model_config.json"
#ocr_model_config = f"{models_path}/{dataset}/model_config.json"

#ocr_model_config = "Models/OCRModels/DergeTenjur/November2023_v2/model_config.json"
ocr_inference = OnnxInference(config_file=model_config, mode="cuda")

In [13]:
test_set_index = f"{dataset_root}/{dataset}/{chckpt}/test_imgs.txt"

with open(test_set_index, "r") as f:
    entries = f.readlines()
    entries = [get_file_name(x) for x in entries]

print(entries[:10])
print(len(entries))

['1-1-20b_5', '52-1-198b_4', '27-1-218a_0', '106-1-54a_4', '89-1-268b_1', '43-1-257b_6', '11-1-204a_4', '51-1-14b_1', '31-1-203b_2', '68-1-68a_5']
51968


In [14]:
#dataset_path = "Data/W23703/Dataset"
#dataset_path = "D:/Datasets/Tibetan/LhasaKanjur/LhasaKanjurProdigy_v1_reviewed"
dataset_path = "D:/Datasets/Tibetan/OCR/LithangKanjur/DatasetFiltered"
#dataset_path = "D:/Datasets/Tibetan/Glomanthang/Glomanthang_January204_merged_bw"
all_images = natsorted(glob(f"{dataset_path}/lines/*.jpg"))
all_labels = natsorted(glob(f"{dataset_path}/transcriptions/*.txt"))

print(f"All Images: {len(all_images)}, All Labels: {len(all_images)}")

test_images = [f"{dataset_path}/lines/{x}.jpg" for x in entries]
test_labels = [f"{dataset_path}/transcriptions/{x}.txt" for x in entries]

print(f"Test Images: {len(test_images)}, Test Labels: {len(test_labels)}")

All Images: 520000, All Labels: 520000
Test Images: 51968, Test Labels: 51968


In [18]:
def preprocess_wylie(label: str) -> str:
    label = label.replace("/ /", "//")
    label = label.replace("/_/", "//")
    label = label.replace("/ ", "/")
    label = label.replace("/_", "/")
    return label

In [31]:
cer_scores = []

for image, label in tqdm(zip(test_images, test_labels), total=len(test_images)):
    test_img = cv2.imread(image)
    test_lbl = read_label(label)
    #test_lbl = converter.toWylie(test_lbl)
    test_lbl = preprocess_wylie(test_lbl)
    test_lbl = process_wylie_label(test_lbl)

    test_img = prepare_ocr_image2(test_img, target_height=ocr_inference._input_height)
    np_img_batch = np.array(test_img)
    np_img_batch = np.expand_dims(np_img_batch, axis=1)

    predicted_text, raw = ocr_inference.run(np_img_batch, replace_tsheg=" ")
    predicted_text = predicted_text[0].replace("/ ", "")
    predicted_text = predicted_text.replace(" ", "§")

    cer_score = cer_scorer.compute(predictions=[predicted_text], references=[test_lbl])
    cer_scores.append(cer_score)


mean_cer = np.mean(cer_scores)

print(f"Mean CER: {mean_cer}")
print(f"Max CER: {max(cer_scores)}")
print(f"Min CER: {min(cer_scores)}")

with open(f"{dataset_root}/{dataset}/{chckpt}/test_set_cer_scores.txt", "w") as f:
    for idx, c_score in enumerate(cer_scores):
            f.write(f"{c_score}, {test_labels[idx]}\n")

 26%|██▌       | 13416/51968 [26:45<1:16:53,  8.36it/s]


KeyboardInterrupt: 

In [30]:
# single line image:
idx = 1234

test_img = cv2.imread(test_images[idx])
test_lbl = read_label(test_labels[idx])
test_lbl = preprocess_wylie(test_lbl)
test_lbl = process_wylie_label(test_lbl)


test_img = prepare_ocr_image2(test_img, target_height=ocr_inference._input_height)
img_batch = np.array(test_img)
img_batch = np.expand_dims(img_batch, axis=1)

predicted_text, raw = ocr_inference.run(img_batch, replace_tsheg=" ")
predicted_text = predicted_text[0].replace("/ ", "")
predicted_text = predicted_text.replace(" ", "§")
print(test_lbl)
print(predicted_text)

cer_score = cer_scorer.compute(predictions=[predicted_text], references=[test_lbl])
print(cer_score)
cer_score = fastwer.score_sent(predicted_text, label, char_level=True)
print(cer_score / len(predicted_text))


du§rmad§du§byung§gyur§pa//bkra§shis§des§ni§skye§dgu§rnams§la§zhi§byed§shog§/bkra§shis§gang§zhig§phan§byed§dam§pa§dag§byed§pa//bsod§nams§bya§byed§'phags§pa'i§skye§bos§mngon§bstod§pa//bcom§ldan§
du§rmad§du§byur§gyur§pa//bkra§shis§des§ni§skye§dgu§rnams§la§zhi§byed§shog§/dkra§shes§gang§zhig§phan§byed§dam§pa§dag§byed§pa//bsod§nams§bya§byed§'phags§pa'i§skye§bos§mngon§pstod§pa//bcom§ldan§
0.026041666666666668
1.2590376963350787


In [10]:
cer_score = cer_scorer.compute(predictions=predicted_text, references=[test_lbl])
print(cer_score)

1.1481481481481481


In [11]:
test_lbl

'འབུལ་ལོ།§།བཞིན་བཟང༌།§ཨ་རོག་ཀྱ།§དེ་དག་ཁྱོད་ཀྱིར་གྱུར་ཅིག§།སྦྱིན་པ་འདིས་ཁྱོད་སེམས་ཀྱི་རྒྱན་དང༌།§སེམས་ཀྱི་ཡོ་བྱད་དང༌།§རྣལ་འབྱོར་གྱི་ཚོགས་དང༌།§དོན་གྱི་མཆོག་ཐོབ་པ་དང༌།'

In [12]:
predicted_text

["'bul lo/ /bzhin bzang / a rog kya/ de dag khyod kyir gyur cig /sbyin pa 'dis khyod sems kyi rgyan dang / sems kyi yo byad dang / rnal 'byor gyi tshogs dang / don gyi mchog thob pa dang /"]

In [18]:
test_lbl = converter.toWylie(test_lbl)

In [19]:
test_lbl

"'bul lo/_/bzhin bzang*/_a rog kya/_de dag khyod kyir gyur cig_/sbyin pa 'dis khyod sems kyi rgyan dang*/_sems kyi yo byad dang*/_rnal 'byor gyi tshogs dang*/_don gyi mchog thob pa dang*/"